# 1 - imports

In [1]:
import os
import json
import math
import re
from math import exp
import random
from collections import Counter
from typing import List, Dict

import numpy as np
from tqdm import tqdm

import torch
torch._dynamo.disable()

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import mean_squared_error

import csv
import pandas as pd
import matplotlib.pyplot as plt

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

import csv
from datetime import datetime

# torch._dynamo.disable()

c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2 - config

In [2]:

CONFIG = {
    'annotations': r"E:\WLASL\wlasl_1000_preproc\train_final.json",
    'data_root': r'E:\WLASL\wlasl_1000_preproc\videos',

    # --- saving ---
    'save_dir': './checkpoints_text2sign_4_wlasl1000_lsmu',
    'save_every': 1,

    # --- training ---
    'batch_size': 24,
    'epochs': 100,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'patience': 10,
    'grad_clip': 1.0,
    'teacher_forcing_rate': 0.7,
    'lambda_vel': 0.1,
    'disable_early_stop': False,

    # --- device ---
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42,

    # --- text ---
    'max_text_len': 32,
    'pad_id': 0,

    # --- landmarks ---
    'max_landmark_len': 20,
    'num_landmarks': 75,
    'landmark_dim': 3,
    'landmark_input_dim': 75 * 3,  # 225

    # --- transformer ---
    'd_model': 512,
    'nhead': 8,
    'num_encoder_layers': 3,
    'num_decoder_layers': 3,
    'dropout': 0.1,
    
    # --- LSMU ---
    'num_handshape': None,
    'num_location': None,
    'num_movement': None,
    'num_orientation': None,

    # weight of LSMU loss
    'lambda_lsmu': 0.3,
    
    "custom_accuracy": {
        "enabled": True,

        # how predicted sequence is compared to GT instances of same gloss
        # "best"  -> min distance across all .npy files of that gloss
        # "mean"  -> mean distance across all .npy files
        "match_mode": "best",

        # how distance is converted to accuracy
        # "exp"        -> exp(-dist / alpha)
        # "threshold"  -> dist <= threshold ? 1 : 0
        # "linear"     -> max(0, 1 - dist / max_dist)
        # "none"       -> raw distance (no accuracy)
        "convert": "exp",

        # exp conversion parameter (higher = more forgiving)
        # typical range: 0.02 – 0.2 if coords are normalized
        "alpha": 0.05,

        # threshold for "threshold" or "linear" mode
        "threshold": 0.1,

        # resampling target
        # usually same as model output length
        "target_frames": 20,

        # whether to weight joints differently
        "use_joint_weights": True,

        # joint weights (length = 75)
        # pose joints lighter, hands heavier
        "joint_weights": {
            "pose": 0.5,     # 33 joints
            "left_hand": 1.0, # 21 joints
            "right_hand": 1.0 # 21 joints
        },

        # cache GT sequences in memory for speed
        "preload_gt": True,

        # compute this metric on validation only
        "compute_on": "val",  # "train" | "val" | "both"

        # log per-gloss breakdown (expensive but informative)
        "log_per_gloss": False
    }
}


os.makedirs(CONFIG['save_dir'], exist_ok=True)
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])

In [3]:
import logging

log_path = os.path.join(CONFIG['save_dir'], "train.log")

logging.basicConfig(
    filename=log_path,
    filemode="a",
    level=logging.INFO,
    format="%(asctime)s | %(message)s"
)

logger = logging.getLogger()

# 3 - tokenizer

In [4]:
class HFTokenizer:
    def __init__(self, texts, min_freq=1, max_vocab=None):

        self.pad_token = "<pad>"
        self.sos_token = "<sos>"
        self.eos_token = "<eos>"
        self.unk_token = "<unk>"
        specials = [self.pad_token, self.sos_token, self.eos_token, self.unk_token]

        self.tokenizer = Tokenizer(WordPiece(unk_token=self.unk_token))
        self.tokenizer.pre_tokenizer = Whitespace()

        if max_vocab is None:
            max_vocab = 5000

        trainer = WordPieceTrainer(
            special_tokens=specials,
            min_frequency=min_freq,
            vocab_size=max_vocab,
        )


        self.tokenizer.train_from_iterator(texts, trainer)


        self.stoi = self.tokenizer.get_vocab()
        self.itos = sorted(self.stoi, key=lambda w: self.stoi[w])


        self.pad_id = self.stoi[self.pad_token]
        self.sos_id = self.stoi[self.sos_token]
        self.eos_id = self.stoi[self.eos_token]
        self.unk_id = self.stoi[self.unk_token]
        
        self.tokenizer.save("tokenizer.json")
        
        # ------- check -----------
        
        print("Vocab size:", self.vocab_size)
        print("Sample vocab:", list(self.stoi.keys())[:20])
        
        # test = texts[:100]
        unk_ratio = sum(self.unk_id in self.encode(t, 32) for t in texts) / len(texts)
        print("UNK ratio:", unk_ratio)

    @property
    def vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def encode(self, text, max_len):
        text = re.sub(r"[^\w\s]", "", text.lower())
        ids = self.tokenizer.encode(text.strip()).ids
        ids = ids[: max_len - 2]
        return [self.sos_id] + ids + [self.eos_id]

    def pad(self, ids, max_len):
        if len(ids) < max_len:
            ids = ids + [self.pad_id] * (max_len - len(ids))
        return ids[:max_len]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == self.eos_id:
                break
            if i in (self.pad_id, self.sos_id):
                continue
            if i >= len(self.itos):
                continue
            out.append(self.itos[i])
        return " ".join(out)


# 4 - dataset class

In [5]:
class TextToSignDataset(Dataset):
    def __init__(
        self,
        annotations_path,
        data_root,
        tokenizer,
        max_text_len=32,
        max_landmark_len=20,
        min_frames=1
    ):
        with open(annotations_path, "r") as f:
            self.annotations = json.load(f)

        self.data_root = data_root
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len
        self.max_landmark_len = max_landmark_len
        self.min_frames = min_frames

        self.samples = []
        print("[Dataset] Scanning annotations...")

        for entry in tqdm(self.annotations):
            class_id = entry.get("id")
            gloss = entry.get("gloss", "").strip()

            if not class_id or not gloss:
                continue

            for inst in entry.get("instances", []):
                vid = inst.get("video_id")
                if not vid:
                    continue

                fname = vid if vid.endswith(".npy") else f"{vid}.npy"

                npy_path = os.path.join(self.data_root, str(class_id), fname)
                

                # print("DEBUG CHECK")
                # print("  data_root :", self.data_root)
                # print("  class_id  :", repr(class_id), type(class_id))
                # print("  gloss     :", repr(gloss))
                # print("  video_id  :", repr(vid))
                # print("  expected  :", npy_path)
                # print("  exists    :", os.path.exists(npy_path))
       

                if os.path.exists(npy_path):
                    self.samples.append({
                        "npy_path": npy_path,
                        "text": gloss,      
                        "class_id": class_id 
                    })
        
        # ============================
        # LOAD LSMU DATA
        # ============================

        self.lsmu_map = {}

        lsmu_path = os.path.join(os.path.dirname(annotations_path), "final_lsmu.json")

        if os.path.exists(lsmu_path):
            with open(lsmu_path, "r") as f:
                lsmu_data = json.load(f)

            for item in lsmu_data:
                word = item["word"]
                self.lsmu_map[word] = item["lsmu"]

            print(f"[Dataset] Loaded LSMU for {len(self.lsmu_map)} glosses")
        else:
            print("[WARNING] LSMU file not found")
        
        # ============================
        # BUILD LABEL MAPS
        # ============================

        self.handshape_map = {}
        self.location_map = {}
        self.movement_map = {}
        self.orientation_map = {}

        def build_map(key):
            values = set()
            for v in self.lsmu_map.values():
                if key in v:
                    values.add(v[key])
            return {val: i for i, val in enumerate(sorted(values))}

        self.handshape_map = build_map("handshape")
        self.location_map = build_map("location")
        self.movement_map = build_map("movement")
        self.orientation_map = build_map("orientation")
        
        print(f"[Dataset] Total usable pairs: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # ---------- Load landmarks ----------
        try:
            lm = np.load(sample["npy_path"])  # (T,75,3)
        except:
            return None

        if lm.ndim != 3 or lm.shape[0] < self.min_frames:
            return None

        lm = lm.astype(np.float32)
        T = lm.shape[0]

        # ---------- Pad / truncate ----------
        if T >= self.max_landmark_len:
            idxs = np.linspace(0, T - 1, self.max_landmark_len, dtype=int)
            lm = lm[idxs]
            lmask = np.ones(self.max_landmark_len, dtype=np.bool_)
        else:
            pad_len = self.max_landmark_len - T
            pad = np.zeros((pad_len, 75, 3), dtype=np.float32)
            lm = np.concatenate([lm, pad], axis=0)
            lmask = np.array([1] * T + [0] * pad_len, dtype=np.bool_)

        # ---------- Flatten landmarks ----------
        lm = lm.reshape(self.max_landmark_len, -1)  # (T, 225)

        # ---------- Tokenize text ----------
        tokens = self.tokenizer.encode(sample["text"], self.max_text_len)
        tokens = self.tokenizer.pad(tokens, self.max_text_len)

        imask = [t != self.tokenizer.pad_id for t in tokens]
        
        # ---------- LSMU ----------
        lsmu = self.lsmu_map.get(sample["text"], None)

        if lsmu is None:
            return None

        try:
            lsmu_labels = {
                "handshape": self.handshape_map[lsmu["handshape"]],
                "location": self.location_map[lsmu["location"]],
                "movement": self.movement_map[lsmu["movement"]],
                "orientation": self.orientation_map[lsmu["orientation"]],
            }
        except KeyError:
            return None

        return {
            "input_ids": torch.tensor(tokens, dtype=torch.long),
            "input_mask": torch.tensor(imask, dtype=torch.bool),
            "landmarks": torch.tensor(lm, dtype=torch.float32),
            "landmark_mask": torch.tensor(lmask, dtype=torch.bool),
            "gloss": sample["text"],
            
            "lsmu_handshape": torch.tensor(lsmu_labels["handshape"], dtype=torch.long),
            "lsmu_location": torch.tensor(lsmu_labels["location"], dtype=torch.long),
            "lsmu_movement": torch.tensor(lsmu_labels["movement"], dtype=torch.long),
            "lsmu_orientation": torch.tensor(lsmu_labels["orientation"], dtype=torch.long),
        }


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "input_mask": torch.stack([b["input_mask"] for b in batch]),
        "landmarks": torch.stack([b["landmarks"] for b in batch]),
        "landmark_mask": torch.stack([b["landmark_mask"] for b in batch]),
        "gloss": [b["gloss"] for b in batch],
        
        "lsmu_handshape": torch.stack([b["lsmu_handshape"] for b in batch]),
        "lsmu_location": torch.stack([b["lsmu_location"] for b in batch]),
        "lsmu_movement": torch.stack([b["lsmu_movement"] for b in batch]),
        "lsmu_orientation": torch.stack([b["lsmu_orientation"] for b in batch]),
    }


# 5 - t2sModel

In [6]:
def generate_square_subsequent_mask(size, device):
    """Generate a square mask for the sequence. The masked positions are set to float('-inf')."""
    mask = torch.triu(torch.ones(size, size, device=device) * float('-inf'), diagonal=1)
    return mask


class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.max_len = max_len
        self.d_model = d_model
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)
    
    def forward(self, length):
        return self.pe[:length]


class TextToSignModel(nn.Module):
    def __init__(self, vocab_size, landmark_dim, cfg: Dict):
        super().__init__()
        d_model = cfg['d_model']

        self.d_model = d_model
        self.landmark_dim = landmark_dim
        self.max_landmark_len = cfg['max_landmark_len']
        self.max_text_len = cfg['max_text_len']

        # ---- Text encoder ----
        # cfg['pad_id'] = Tokenizer.pad_id
        self.tok_embed = nn.Embedding(
            vocab_size, d_model, padding_idx=cfg['pad_id']
        )
        self.text_pos = PositionalEmbedding(self.max_text_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=cfg['nhead'],
            dim_feedforward=d_model * 4,
            dropout=cfg['dropout'],
            batch_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=cfg['num_encoder_layers']
        )

        # ---- Landmark side ----
        self.landmark_in_proj = nn.Linear(landmark_dim, d_model)
        self.start_frame = nn.Parameter(torch.zeros(1, 1, d_model))
        self.landmark_pos = PositionalEmbedding(self.max_landmark_len, d_model)

        # ---- Decoder ----
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=cfg['nhead'],
            dim_feedforward=d_model * 4,
            dropout=cfg['dropout'],
            batch_first=True,
            activation='gelu'
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=cfg['num_decoder_layers']
        )

        # ---- Output ----
        self.out_proj = nn.Linear(d_model, landmark_dim)

        self._init_weights()
        
        # ---- LSMU Head ----
        self.lsmu_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(cfg['dropout'])
        )

        self.handshape_cls = nn.Linear(d_model, cfg['num_handshape'])
        self.location_cls = nn.Linear(d_model, cfg['num_location'])
        self.movement_cls = nn.Linear(d_model, cfg['num_movement'])
        self.orientation_cls = nn.Linear(d_model, cfg['num_orientation'])

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode_text(self, input_ids, input_mask):
        emb = self.tok_embed(input_ids)  # (B, T_text, d)
        pos = self.text_pos(emb.size(1)).unsqueeze(0)
        src = emb + pos

        src_key_padding_mask = ~input_mask
        memory = self.encoder(
            src,
            src_key_padding_mask=src_key_padding_mask
        )
        return memory, src_key_padding_mask

    def forward(
        self,
        input_ids,
        input_mask,
        tgt_landmarks=None,
        teacher_forcing_ratio=0.7
    ):
        """
        input_ids      : (B, T_text)
        tgt_landmarks  : (B, T_landmark, landmark_dim)
        return         : (B, T_landmark, landmark_dim)
        """

        B = input_ids.size(0)
        device = input_ids.device
        T = self.max_landmark_len

        memory, src_key_padding_mask = self.encode_text(input_ids, input_mask)
        
        # ============================
        # LSMU PREDICTION (from encoder)
        # ============================

        # mean pooling over text tokens
        text_feat = memory.mean(dim=1)  # (B, d_model)

        lsmu_feat = self.lsmu_head(text_feat)

        lsmu_preds = {
            "handshape": self.handshape_cls(lsmu_feat),
            "location": self.location_cls(lsmu_feat),
            "movement": self.movement_cls(lsmu_feat),
            "orientation": self.orientation_cls(lsmu_feat),
        }

        # ---- Decoder input buffer ----
        prev = self.start_frame.expand(B, 1, self.d_model)
        dec_inputs = []

        pos = self.landmark_pos(T).unsqueeze(0)

        for t in range(T):
            tgt = torch.cat(dec_inputs + [prev], dim=1)
            tgt = tgt + pos[:, :tgt.size(1), :]

            tgt_mask = generate_square_subsequent_mask(tgt.size(1), device)

            dec_out = self.decoder(
                tgt,
                memory,
                tgt_mask=tgt_mask,
                memory_key_padding_mask=src_key_padding_mask
            )

            last_hidden = dec_out[:, -1:, :]  # (B,1,d)
            pred = self.out_proj(last_hidden)

            dec_inputs.append(last_hidden)

            if tgt_landmarks is not None and random.random() < teacher_forcing_ratio:
                prev = self.landmark_in_proj(tgt_landmarks[:, t:t+1, :])
            else:
                prev = self.landmark_in_proj(pred)

        dec_feats = torch.cat(dec_inputs, dim=1)
        preds = self.out_proj(dec_feats)

        return preds, lsmu_preds


# 6 - loss

In [7]:
class MaskedSmoothL1Loss(nn.Module):
    def __init__(self, beta=1.0):
        super().__init__()
        self.beta = beta

    def forward(self, pred, target, mask):
        """
        pred   : (B, T, D)
        target : (B, T, D)
        mask   : (B, T)  -> 1 for valid frames, 0 for padding
        """

        # ensure float mask
        mask = mask.float()

        loss = F.smooth_l1_loss(
            pred,
            target,
            reduction='none',
            beta=self.beta
        )  # (B, T, D)

        loss = loss.mean(dim=-1)        # (B, T)
        loss = loss * mask              # mask padding frames

        return loss.sum() / (mask.sum() + 1e-6)


In [8]:
def velocity_loss(pred, mask):
    """
    pred : (B, T, D)
    mask : (B, T)   -> 1 for valid frames
    """

    vel = pred[:, 1:] - pred[:, :-1]   # (B, T-1, D)
    vel_mask = mask[:, 1:].float()     # (B, T-1)

    loss = vel.abs().mean(dim=-1)      # (B, T-1)
    loss = loss * vel_mask

    return loss.sum() / (vel_mask.sum() + 1e-6)


# 7 - metrics

In [9]:
def mpjpe(pred, gt, mask):
    """
    Mean Per-Joint Position Error (flattened landmarks)

    pred, gt : (B, T, D=225)
    mask     : (B, T)
    """
    mask = mask.float()

    diff = pred - gt
    dist = torch.norm(diff, dim=-1)   # (B, T)

    dist = dist * mask
    return dist.sum() / (mask.sum() + 1e-6)


def velocity_error(pred, gt, mask):
    """
    Frame-to-frame velocity error
    """

    v_pred = pred[:, 1:] - pred[:, :-1]
    v_gt   = gt[:, 1:]   - gt[:, :-1]

    vel_mask = mask[:, 1:].float()

    err = torch.norm(v_pred - v_gt, dim=-1)  # (B, T-1)
    err = err * vel_mask

    return err.sum() / (vel_mask.sum() + 1e-6)


# with torch.no_grad():
#     preds = model(...)
#     val_mpjpe = mpjpe(preds, gt, landmark_mask)
#     val_vel   = velocity_error(preds, gt, landmark_mask)


### Accuracy Metric

In [10]:
# ---------
def resample_sequence(arr, target_T):
    """Resample/truncate arr (T_gt, K, 3) -> target_T frames
       using linear index sampling (no interpolation)."""
    T_gt = arr.shape[0]
    if T_gt == target_T:
        return arr
    if T_gt < 1:
        raise ValueError("Empty sequence")
    idxs = np.linspace(0, T_gt - 1, target_T, dtype=int)
    return arr[idxs]

def mean_joint_distance(pred, gt, joint_weights=None):
    """pred,gt: (T, K, 3). Return scalar mean Euclidean distance (frames x joints averaged).
       Optional joint_weights: (K,) to weight joints (hands > pose etc)"""
    diff = pred - gt                           # (T, K, 3)
    per_joint_frame = np.linalg.norm(diff, axis=-1)  # (T, K)
    if joint_weights is not None:
        jw = np.asarray(joint_weights).reshape(1, -1)
        per_joint_frame = per_joint_frame * jw
        return per_joint_frame.sum() / jw.sum() / per_joint_frame.shape[0]
    return per_joint_frame.mean()

# ------- primary metric -----
def compute_gloss_match_score(
    pred,                 # (T, K, 3) predicted sequence (numpy)
    gt_paths,             # list[str] paths to .npy ground-truth sequences for the gloss
    target_T=None,        # if None, pred.shape[0] used
    mode="best",          # "best" -> min distance across instances, "mean" -> avg across instances
    convert="exp",        # convert distance -> accuracy: "exp" | "threshold" | "linear" | "none"
    alpha=5.0,            # for exp: acc = exp(-dist/alpha)
    threshold=0.5,        # for threshold: dist <= threshold -> acc=1 else 0
    joint_weights=None,   # optional (K,) weights array
    preload_cache=None    # dict path->ndarray to avoid reloading disk every call (optional)
):
    """
    Returns:
      score, best_dist, per_instance_dists
      - score: accuracy-like in [0,1] (unless convert="none")
      - best_dist: the chosen distance (min or mean)
      - per_instance_dists: list of distances for each gt instance
    """
    if target_T is None:
        target_T = pred.shape[0]

    per_instance = []

    for p in gt_paths:
        if preload_cache is not None and p in preload_cache:
            gt_arr = preload_cache[p]
        else:
            gt_arr = np.load(p)   # (T_gt, K, 3)

        # resample/truncate to target_T
        try:
            gt_seq = resample_sequence(gt_arr, target_T)
        except Exception as e:
            continue

        # compute distance
        d = mean_joint_distance(pred, gt_seq, joint_weights)
        per_instance.append(float(d))

    if len(per_instance) == 0:
        return None, None, []

    if mode == "best":
        chosen = float(np.min(per_instance))
    else:  # "mean"
        chosen = float(np.mean(per_instance))

    # convert to accuracy-like
    if convert == "exp":
        score = float(np.exp(-chosen / float(alpha)))
    elif convert == "threshold":
        score = 1.0 if chosen <= threshold else 0.0
    elif convert == "linear":
        # map dist 0->max_dist to [1->0]
        max_dist = max(threshold, 1.0)
        score = max(0.0, 1.0 - chosen / max_dist)
    elif convert == "none":
        score = chosen
    else:
        raise ValueError("unknown convert")

    return score, chosen, per_instance

# ---- build gloss->paths map from annotations -----
def build_gloss_to_paths(annotations_path, data_root):
    # annotations expected to be list of {"id": "1", "gloss": "about", "instances":[{"video_id":"0001"}, ...]}
    with open(annotations_path, "r", encoding="utf-8") as f:
        ann = json.load(f)

    gloss_map = {}
    for entry in ann:
        class_id = str(entry.get("id"))
        gloss = entry.get("gloss")
        if class_id is None or gloss is None:
            continue
        paths = []
        for inst in entry.get("instances", []):
            vid = inst.get("video_id")
            if not vid:
                continue
            fname = vid if vid.endswith(".npy") else f"{vid}.npy"
            p = os.path.join(data_root, class_id, fname)
            if os.path.exists(p):
                paths.append(p)
        if len(paths) > 0:
            gloss_map[gloss] = paths
    return gloss_map


LSMU Loss

In [11]:
class LSMULoss(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.weights = weights or {
            "handshape": 1.0,
            "location": 1.0,
            "movement": 1.0,
            "orientation": 1.0
        }

    def forward(self, preds, targets):
        loss = 0

        loss += self.weights["handshape"] * self.ce(preds["handshape"], targets["handshape"])
        loss += self.weights["location"]  * self.ce(preds["location"], targets["location"])
        loss += self.weights["movement"]  * self.ce(preds["movement"], targets["movement"])
        loss += self.weights["orientation"] * self.ce(preds["orientation"], targets["orientation"])

        return loss

## utils

In [12]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import os
import numpy as np

def save_confusion_matrix(epoch, y_true, y_pred, save_dir, labels=None):

    if len(y_true) == 0:
        print("[Confusion] No data to compute.")
        return

    os.makedirs(save_dir, exist_ok=True)

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    np.save(os.path.join(save_dir, f"confusion_epoch_{epoch}.npy"), cm)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

    fig, ax = plt.subplots(figsize=(10, 8))
    disp.plot(ax=ax, cmap="magma", colorbar=True)

    plt.title(f"Confusion Matrix (Epoch {epoch})")
    plt.tight_layout()

    plt.savefig(os.path.join(save_dir, f"confusion_epoch_{epoch}.png"))
    plt.close()

    print(f"[Confusion] Saved epoch {epoch}")

In [13]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

plt.rcParams.update({
    "figure.figsize": (8,5),
    "font.size": 12,
    "font.family": "serif",
    "axes.grid": True,
    "axes.linewidth": 1.0,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 150
})

def plot_learning_curves_from_csv(metrics_csv):
    df = pd.read_csv(metrics_csv)
    # smoothing for presentation (optional)
    df['train_loss_smooth'] = df['train_loss'].rolling(3, min_periods=1).mean()
    df['val_loss_smooth'] = df['val_loss'].rolling(3, min_periods=1).mean()

    fig, ax = plt.subplots()
    ax.plot(df['epoch'], df['train_loss_smooth'], label='Train loss', linewidth=2)
    ax.plot(df['epoch'], df['val_loss_smooth'], label='Val loss', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Validation Loss')
    ax.legend()
    ax.grid(alpha=0.25)
    # annotate best val epoch
    best_idx = df['val_loss'].idxmin()
    best_epoch = int(df.loc[best_idx, 'epoch'])
    best_val = df.loc[best_idx, 'val_loss']
    ax.scatter([best_epoch], [best_val], color='k')
    ax.annotate(f'best val: epoch {best_epoch}\n{best_val:.4f}', xy=(best_epoch, best_val), xytext=(best_epoch+1, best_val*1.05),
                arrowprops=dict(arrowstyle="->", lw=1.0))
    plt.tight_layout()
    return fig

def plot_metric_subplots(metrics_csv):
    df = pd.read_csv(metrics_csv)
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    axs = axs.ravel()
    axs[0].plot(df['epoch'], df['train_loss'], label='train', linewidth=1.8)
    axs[0].plot(df['epoch'], df['val_loss'], label='val', linewidth=1.8)
    axs[0].set_title('Loss')

    if 'val_mpjpe' in df.columns:
        axs[1].plot(df['epoch'], df['val_mpjpe'], linewidth=1.6)
        axs[1].set_title('MPJPE (val)')

    if 'val_velocity_error' in df.columns:
        axs[2].plot(df['epoch'], df['val_velocity_error'], linewidth=1.6)
        axs[2].set_title('Velocity Error (val)')

    axs[3].plot(df['epoch'], df.get('teacher_forcing_ratio', np.zeros(len(df))), linewidth=1.6)
    axs[3].set_title('Teacher forcing ratio (if present)')

    for ax in axs:
        ax.grid(alpha=0.2)
    plt.tight_layout()
    return fig

def plot_confusion_matrix(y_true, y_pred, labels, normalize=True, figsize=(8,6), cmap='viridis'):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if normalize:
        cm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-12)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='.2f' if normalize else 'd', cmap=cmap, ax=ax, cbar=True,
                xticklabels=labels, yticklabels=labels)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix' + (' (normalized)' if normalize else ''))
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    return fig

In [14]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def plot_grad_norms(model):
    norms = []
    names = []
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        gnorm = p.grad.data.norm(2).item()
        norms.append(gnorm)
        names.append(name)
    fig, ax = plt.subplots(figsize=(8,4))
    ax.barh(range(len(norms)), norms, color='tab:blue')
    ax.set_yticks(range(len(norms)))
    ax.set_yticklabels(names)
    ax.set_title('Parameter gradient norms')
    plt.tight_layout()
    return fig

def plot_weight_histograms(model, bins=60):
    all_weights = []
    for p in model.parameters():
        all_weights.append(p.data.cpu().numpy().ravel())
    all_weights = np.concatenate(all_weights)
    fig, ax = plt.subplots(figsize=(6,4))
    ax.hist(all_weights, bins=bins)
    ax.set_title('Weight distribution (all layers concatenated)')
    plt.tight_layout()
    return fig

def plot_embeddings_tsne(embeddings, labels=None, perplexity=30, n_iter=1000):
    """
    embeddings: np.array of shape (N, D)
    labels: optional, shape (N,)
    """
    ts = TSNE(n_components=2, perplexity=perplexity, n_iter=n_iter, init='pca', random_state=42)
    z = ts.fit_transform(embeddings)
    fig, ax = plt.subplots(figsize=(7,6))
    if labels is None:
        ax.scatter(z[:,0], z[:,1], s=10, alpha=0.8)
    else:
        for ll in np.unique(labels):
            idx = labels == ll
            ax.scatter(z[idx,0], z[idx,1], label=str(ll), s=10, alpha=0.8)
        ax.legend(markerscale=2)
    ax.set_title('t-SNE of embeddings')
    plt.tight_layout()
    return fig

In [15]:
from sklearn.metrics import precision_recall_fscore_support

def save_classification_report(epoch, y_true, y_pred, save_dir, labels):
    from sklearn.metrics import classification_report
    import json, os

    os.makedirs(save_dir, exist_ok=True)

    # Ensure integer labels
    label_ids = list(range(len(labels)))

    report = classification_report(
        y_true,
        y_pred,
        labels=label_ids,         
        target_names=labels,      
        output_dict=True,
        zero_division=0
    )

    path = os.path.join(save_dir, f"classification_report_epoch_{epoch}.json")
    with open(path, "w") as f:
        json.dump(report, f, indent=4)

    print(f"[Saved] classification report epoch {epoch}")

In [16]:
def generate_research_plots(metrics_csv, metrics_dir):
    import pandas as pd
    import matplotlib.pyplot as plt
    import os

    df = pd.read_csv(metrics_csv)

    plots_dir = os.path.join(metrics_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    # -------- LOSS CURVE --------
    plt.figure()
    plt.plot(df['epoch'], df['train_loss'], label='Train', linewidth=2)
    plt.plot(df['epoch'], df['val_loss'], label='Validation', linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "loss_curve.png"), dpi=300)
    plt.close()

    # -------- MPJPE --------
    plt.figure()
    plt.plot(df['epoch'], df['val_mpjpe'], linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("MPJPE")
    plt.title("Validation MPJPE")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "mpjpe.png"), dpi=300)
    plt.close()

    # -------- VELOCITY ERROR --------
    plt.figure()
    plt.plot(df['epoch'], df['val_velocity_error'], linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Velocity Error")
    plt.title("Validation Velocity Error")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "velocity_error.png"), dpi=300)
    plt.close()

    # -------- TEACHER FORCING --------
    plt.figure()
    plt.plot(df['epoch'], df['teacher_forcing_ratio'], linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("TF Ratio")
    plt.title("Teacher Forcing Schedule")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "teacher_forcing.png"), dpi=300)
    plt.close()

    print(f"[Plots Saved] → {plots_dir}")

In [17]:
def analyze_confusion_trends(metrics_dir):
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    cm_dir = os.path.join(metrics_dir, "confusion")

    files = sorted([f for f in os.listdir(cm_dir) if f.endswith(".npy")])

    accuracies = []

    for f in files:
        cm = np.load(os.path.join(cm_dir, f))
        acc = np.trace(cm) / (cm.sum() + 1e-9)
        accuracies.append(acc)

    plt.figure()
    plt.plot(range(1, len(accuracies)+1), accuracies, linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Confusion Accuracy")
    plt.title("Classification Accuracy Trend (from CM)")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    save_path = os.path.join(metrics_dir, "plots", "confusion_accuracy.png")
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"[Saved] {save_path}")

In [18]:
def save_classification_report(epoch, y_true, y_pred, save_dir, labels):
    from sklearn.metrics import classification_report
    import json, os

    os.makedirs(save_dir, exist_ok=True)

    # Convert labels → indices if needed
    if isinstance(y_true[0], str):
        label_to_id = {l: i for i, l in enumerate(labels)}
        y_true = [label_to_id[y] for y in y_true]
        y_pred = [label_to_id[y] for y in y_pred]

    label_ids = list(range(len(labels)))

    report = classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=labels,
        output_dict=True,
        zero_division=0
    )

    with open(os.path.join(save_dir, f"classification_report_epoch_{epoch}.json"), "w") as f:
        json.dump(report, f, indent=4)

# 8 - training and eval

In [19]:
def landmark_accuracy(pred, gt, mask, alpha=5.0):
    """
    Soft landmark accuracy for 3D landmarks

    pred, gt : (B, T, D=225)
    mask     : (B, T)
    alpha    : distance scaling factor
    """

    mask = mask.float()
    if mask.sum() == 0:
        return None

    # reshape to (B, T, 75, 3)
    B, T, D = pred.shape
    pred = pred.view(B, T, -1, 3)
    gt   = gt.view(B, T, -1, 3)

    # per-joint distance
    dist = torch.norm(pred - gt, dim=-1)   # (B, T, 75)

    # mean over joints
    dist = dist.mean(dim=-1)                # (B, T)

    # mask padding frames
    dist = dist * mask

    mean_dist = dist.sum() / (mask.sum() + 1e-6)

    acc = torch.exp(-mean_dist / alpha)
    return acc.item()


In [20]:
def train_one_epoch(model, dataloader, optimizer, criterion, lsmu_criterion, cfg, device, tf_ratio):
    print("\n[Train] Starting epoch...")
    model.train()
    
    cfg['lambda_lsmu'] = 0.2
    total_loss = 0.0
    total_weight = 0.0  

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    for batch in tqdm(dataloader, desc="Train", leave=False):
        if batch is None:
            continue

        for k in batch:
            if isinstance(batch[k], torch.Tensor):
                batch[k] = batch[k].to(device)

        input_ids = batch['input_ids']
        input_mask = batch['input_mask']
        landmarks = batch['landmarks']
        landmark_mask = batch['landmark_mask']
        lsmu_targets = {
            "handshape": batch["lsmu_handshape"],
            "location": batch["lsmu_location"],
            "movement": batch["lsmu_movement"],
            "orientation": batch["lsmu_orientation"],
        }

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            preds, lsmu_preds = model(
                input_ids,
                input_mask,
                tgt_landmarks=landmarks,
                teacher_forcing_ratio=tf_ratio
            )

            pose_loss = criterion(preds, landmarks, landmark_mask)
            vel_loss = velocity_loss(preds, landmark_mask)
            loss_lsmu = lsmu_criterion(lsmu_preds, lsmu_targets)

            loss = (
                pose_loss
                + cfg['lambda_vel'] * vel_loss
                + cfg['lambda_lsmu'] * loss_lsmu
            )
            

        if 'input_mask' in batch:
            batch_weight = input_mask.float().sum().item()
            if batch_weight == 0:
                continue
        else:
            batch_weight = input_ids.shape[0]

        # backward
        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            cfg.get('grad_clip', 1.0)
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * batch_weight
        total_weight += batch_weight

    avg_loss = total_loss / max(1.0, total_weight)

    print(f"[Train] Epoch complete. Avg Loss: {avg_loss:.6f}")
    return avg_loss

@torch.no_grad()
def validate(model, dataloader, criterion, lsmu_criterion, cfg, device, gloss_map, preload_cache=None):
    print("\n[Val] Running validation...")
    model.eval()

    total_loss = 0.0
    total_pose = 0.0
    total_vel = 0.0
    total_mpjpe = 0.0
    total_vel_err = 0.0
    
    total_acc = 0.0
    acc_count = 0
    
    total_weight = 0.0 
    
    all_preds = []
    all_targets = []
    all_gloss_pred = []
    all_gloss_true = []
    
    lsmu_correct = 0
    lsmu_total = 0  

    with torch.no_grad(): 
        for batch in tqdm(dataloader, desc="Val", leave=False):
            if batch is None:
                continue

            for k in batch:
                if isinstance(batch[k], torch.Tensor):
                    batch[k] = batch[k].to(device)

            input_ids = batch['input_ids']
            input_mask = batch['input_mask']
            landmarks = batch['landmarks']
            landmark_mask = batch['landmark_mask']
            gloss = batch["gloss"]
            lsmu_targets = {
                "handshape": batch["lsmu_handshape"],
                "location": batch["lsmu_location"],
                "movement": batch["lsmu_movement"],
                "orientation": batch["lsmu_orientation"],
            }

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                preds, lsmu_preds = model(
                    input_ids,
                    input_mask,
                    tgt_landmarks=None,
                    teacher_forcing_ratio=0.0
                )

                pose_loss = criterion(preds, landmarks, landmark_mask)
                vel_loss = velocity_loss(preds, landmark_mask)
                loss_lsmu = lsmu_criterion(lsmu_preds, lsmu_targets)

                loss = (
                    pose_loss
                    + cfg['lambda_vel'] * vel_loss
                    + cfg['lambda_lsmu'] * loss_lsmu
                )
                
                
            pred_labels = {
                k: torch.argmax(v, dim=1)
                for k, v in lsmu_preds.items()
            }

            for k in pred_labels:
                lsmu_correct += (pred_labels[k] == lsmu_targets[k]).sum().item()
                lsmu_total += lsmu_targets[k].numel()

            if 'input_mask' in batch:
                batch_weight = input_mask.float().sum().item()
            else:
                batch_weight = input_ids.shape[0]

            total_loss += loss.item() * batch_weight
            total_pose += pose_loss.item() * batch_weight
            total_vel += vel_loss.item() * batch_weight

            total_mpjpe += mpjpe(preds, landmarks, landmark_mask).item() * batch_weight
            total_vel_err += velocity_error(preds, landmarks, landmark_mask).item() * batch_weight

            total_weight += batch_weight

            # -------- store predictions --------
            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(landmarks.detach().cpu().numpy())
            
            # -------- gloss accuracy --------
            B = preds.shape[0]

            for i in range(B):
                gloss_i = gloss[i]

                if gloss_i not in gloss_map:
                    continue

                pred_np = preds[i].detach().cpu().numpy()

                best_score = -1
                best_gloss = None

                for gloss_candidate, gt_paths in gloss_map.items():
                    score, _, _ = compute_gloss_match_score(
                        pred=pred_np.reshape(pred_np.shape[0], -1, 3),
                        gt_paths=gt_paths,
                        target_T=pred_np.shape[0],
                        mode="best",
                        convert="exp",
                        alpha=cfg.get("acc_alpha", 5.0),
                        joint_weights=cfg.get("joint_weights"),
                        preload_cache=preload_cache
                    )

                    if score is not None and score > best_score:
                        best_score = score
                        best_gloss = gloss_candidate

                if best_score is not None:
                    total_acc += best_score
                    acc_count += 1
                    
                if best_gloss is not None:
                    all_gloss_pred.append(best_gloss)
                    all_gloss_true.append(gloss_i)

    # -------- concatenate predictions --------
    if len(all_preds) > 0:
        all_preds = np.concatenate(all_preds, axis=0)
        all_targets = np.concatenate(all_targets, axis=0)
    else:
        all_preds = None
        all_targets = None

    metrics = {
        "val_loss": total_loss / max(1.0, total_weight),
        "val_pose_loss": total_pose / max(1.0, total_weight),
        "val_velocity_loss": total_vel / max(1.0, total_weight),
        "val_mpjpe": total_mpjpe / max(1.0, total_weight),
        "val_velocity_error": total_vel_err / max(1.0, total_weight),
        "val_gloss_acc": total_acc / max(1, acc_count),
        
        "predictions": all_preds,
        "targets": all_targets,
        "gloss_pred": all_gloss_pred,
        "gloss_true": all_gloss_true,
        "val_lsmu_acc": lsmu_correct / max(1, lsmu_total),
    }


    msg = (
        f"[Val] Loss: {metrics['val_loss']:.6f} | "
        f"Pose: {metrics['val_pose_loss']:.6f} | "
        f"Vel: {metrics['val_velocity_loss']:.6f} | "
        f"MPJPE: {metrics['val_mpjpe']:.4f} | "
        f"VelErr: {metrics['val_velocity_error']:.4f} | "
        f"GlossAcc: {metrics['val_gloss_acc']:.4f} | "
        f"LSMU: {metrics.get('val_lsmu_acc', 0.0):.4f} | "
    )

    print(msg)        
    logger.info(msg) 

    return metrics

In [21]:
@torch.no_grad()
def generate_from_text(model, tokenizer: HFTokenizer, text: str, cfg, device):
    model.eval()

    # ---- tokenize ----
    tokens = tokenizer.encode(text, max_len=cfg['max_text_len'])
    tokens = tokenizer.pad(tokens, cfg['max_text_len'])

    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    input_mask = torch.tensor(
        [[t != tokenizer.pad_id for t in tokens]],
        dtype=torch.bool
    ).to(device)

    # ---- inference ----
    preds, lsmu_preds = model(
        input_ids,
        input_mask,
        tgt_landmarks=None,
        teacher_forcing_ratio=0.0
    )  # (1, T, D)

    preds = preds.squeeze(0).cpu().numpy()  # (T, 225)

    # preds = preds.reshape(cfg['max_landmark_len'], 75, 3)
    
    # decode LSMU predictions
    lsmu_out = {
        k: torch.argmax(v, dim=1).item()
        for k, v in lsmu_preds.items()
    }

    return preds, lsmu_out

# 9 - main

In [22]:
cfg_init = CONFIG

In [23]:


def main(cfg):
    import os, json, csv, math
    import numpy as np
    import torch
    from torch.utils.data import DataLoader, random_split
    from collections import Counter

    device = torch.device(cfg['device'])
    print(f"[Run] device: {device}")

    # ------------------ dirs ------------------
    os.makedirs(cfg['save_dir'], exist_ok=True)
    metrics_dir = os.path.join(cfg['save_dir'], "metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    metrics_csv = os.path.join(metrics_dir, "epoch_metrics.csv")
    if not os.path.exists(metrics_csv):
        with open(metrics_csv, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "val_pose_loss",
                "val_velocity_loss",
                "val_mpjpe",
                "val_velocity_error",
                "teacher_forcing_ratio",
                "lr",
                "val_lsmu_acc",
            ])

    # ------------------ tokenizer ------------------
    with open(cfg['annotations'], 'r') as f:
        ann = json.load(f)

    texts = [e.get('gloss', '').strip() for e in ann if e.get('gloss', '').strip()]
    tokenizer = HFTokenizer(texts, min_freq=1)
    cfg['pad_id'] = tokenizer.pad_id
    print(f"[Vocab] size: {tokenizer.vocab_size}")

    # ------------------ dataset ------------------
    dataset = TextToSignDataset(
        cfg['annotations'],
        cfg['data_root'],
        tokenizer,
        max_text_len=cfg['max_text_len'],
        max_landmark_len=cfg['max_landmark_len']
    )
    
    # ============================
    # LSMU CONFIG (IMPORTANT)
    # ============================

    cfg['num_handshape'] = len(dataset.handshape_map)
    cfg['num_location'] = len(dataset.location_map)
    cfg['num_movement'] = len(dataset.movement_map)
    cfg['num_orientation'] = len(dataset.orientation_map)

    print("[LSMU]")
    print(" handshape:", cfg['num_handshape'])
    print(" location :", cfg['num_location'])
    print(" movement :", cfg['num_movement'])
    print(" orientation:", cfg['num_orientation'])

    # ------------------ dataset split (reproducible) ------------------
    total = len(dataset)
    n_train = int(total * 0.7)
    n_val   = int(total * 0.2)
    n_test  = total - n_train - n_val

    train_ds, val_ds, test_ds = random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )

    print(f"[Split] train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, collate_fn=collate_fn)

    # ------------------ landmark dim ------------------
    sample_item = next(s for s in dataset if s is not None)
    landmark_dim = sample_item['landmarks'].shape[1]
    print(f"[Landmark dim] {landmark_dim}")

    # ------------------ model ------------------
    model = TextToSignModel(
        vocab_size=tokenizer.vocab_size,
        landmark_dim=landmark_dim,
        cfg=cfg
    ).to(device)

    print(f"[Model] params: {sum(p.numel() for p in model.parameters()):,}")

    # ------------------ optimizer & loss ------------------
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = MaskedSmoothL1Loss(beta=1.0)
    lsmu_criterion = LSMULoss()

    best_val_loss = float('inf')
    patience = cfg['patience']
    epochs_no_improve = 0

    # ------------------ gloss label mapping ------------------
    gloss_map = build_gloss_to_paths(cfg["annotations"], cfg["data_root"])
    label_list = list(gloss_map.keys())
    label_to_id = {l: i for i, l in enumerate(label_list)}

    # ------------------ preload ------------------
    preload_cache = {}
    for gloss, paths in gloss_map.items():
        for p in paths:
            preload_cache[p] = np.load(p)

    # ------------------ helper: classification report ------------------
    def save_classification_report(epoch, y_true, y_pred, save_dir):
        from sklearn.metrics import classification_report
        import json, os

        os.makedirs(save_dir, exist_ok=True)

        # convert string labels → ids
        if isinstance(y_true[0], str):
            y_true = [label_to_id[y] for y in y_true]
            y_pred = [label_to_id[y] for y in y_pred]

        report = classification_report(
            y_true,
            y_pred,
            labels=list(range(len(label_list))),
            target_names=label_list,
            output_dict=True,
            zero_division=0
        )

        with open(os.path.join(save_dir, f"classification_report_epoch_{epoch}.json"), "w") as f:
            json.dump(report, f, indent=4)

    # ------------------ training loop ------------------
    for epoch in range(cfg['epochs']):
        print(f"\n=== Epoch {epoch+1}/{cfg['epochs']} ===")

        tf_ratio = max(0.05, cfg['teacher_forcing_rate'] * (1 - epoch / cfg['epochs']))

        train_loss = train_one_epoch(
            model, train_loader, optimizer,
            criterion, lsmu_criterion, cfg, device, tf_ratio
        )

        val_metrics = validate(
            model, val_loader, criterion, lsmu_criterion, 
            cfg, device, gloss_map, preload_cache
        )

        # ------------------ debug ------------------
        unique_gt = len(set(val_metrics["gloss_true"]))
        unique_pred = len(set(val_metrics["gloss_pred"]))

        print(f"Unique GT classes: {unique_gt}")
        print(f"Unique Pred classes: {unique_pred}")
        print(f"Total labels: {len(label_list)}")

        if unique_pred <= 2:
            print("[WARNING]: Model collapsing to very few classes!")

        # ------------------ confusion ------------------
        save_confusion_matrix(
            epoch=epoch + 1,
            y_true=val_metrics["gloss_true"],
            y_pred=val_metrics["gloss_pred"],
            save_dir=os.path.join(metrics_dir, "confusion"),
            labels=label_list
        )

        # ------------------ classification report ------------------
        save_classification_report(
            epoch=epoch + 1,
            y_true=val_metrics["gloss_true"],
            y_pred=val_metrics["gloss_pred"],
            save_dir=os.path.join(metrics_dir, "reports")
        )

        # ------------------ log CSV ------------------
        with open(metrics_csv, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch + 1,
                train_loss,
                val_metrics["val_loss"],
                val_metrics["val_pose_loss"],
                val_metrics["val_velocity_loss"],
                val_metrics["val_mpjpe"],
                val_metrics["val_velocity_error"],
                tf_ratio,
                optimizer.param_groups[0]["lr"],
                val_metrics.get("val_lsmu_acc", 0.0),
            ])

        # ------------------ save checkpoint ------------------
        if (epoch + 1) % cfg['save_every'] == 0:
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "cfg": cfg,
                "tokenizer_itos": tokenizer.itos
            }, os.path.join(cfg['save_dir'], f"text2sign_epoch_{epoch+1}.pth"))

        # ------------------ early stopping ------------------
        val_loss = float(val_metrics["val_loss"])

        if not math.isnan(val_loss):
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0

                torch.save({
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "cfg": cfg,
                    "tokenizer_itos": tokenizer.itos
                }, os.path.join(cfg['save_dir'], "best_text2sign.pth"))

                print("[Saved] best model")

            else:
                epochs_no_improve += 1
                print(f"[Early Stop] {epochs_no_improve}/{patience}")

        if epochs_no_improve >= patience:
            print(f"== Early stopping at epoch {epoch+1} ==")
            break

    # ------------------ testing ------------------
    print("\n===== Final Test Evaluation =====")

    checkpoint = torch.load(os.path.join(cfg['save_dir'], "best_text2sign.pth"), map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    test_metrics = validate(
        model,
        test_loader,
        criterion,
        cfg,
        device,
        gloss_map,
        preload_cache
    )

    print("\n[Test Results]")
    print(f"Test Loss: {test_metrics['val_loss']:.4f}")
    print(f"Test MPJPE: {test_metrics['val_mpjpe']:.4f}")

    # ------------------ plots ------------------
    print("\n[Post-processing] Generating research plots...")
    generate_research_plots(metrics_csv, metrics_dir)
    analyze_confusion_trends(metrics_dir)

    print("[Done] All plots + reports saved.")

In [24]:
# import torch
# print(torch.__version__)
# print(torch.version.cuda)

# print(batch.keys)

In [ ]:
main(CONFIG)

[Run] device: cuda
Vocab size: 2103
Sample vocab: ['deliver', 'fut', 'enormous', 'lazy', '##ang', 'expert', 'halloween', '##ective', '##istry', 'gas', '##ckey', 'both', '##glass', 'general', 'bus', 'amp', 'king', '##cid', 'award', '##brate']
UNK ratio: 0.0
[Vocab] size: 2103
[Dataset] Scanning annotations...


100%|██████████| 1000/1000 [00:02<00:00, 340.59it/s]


[Dataset] Loaded LSMU for 996 glosses
[Dataset] Total usable pairs: 4204
[LSMU]
 handshape: 4
 location : 4
 movement : 2
 orientation: 3
[Split] train=2942, val=840, test=422
[Landmark dim] 225
[Model] params: 23,646,958

=== Epoch 1/100 ===

[Train] Starting epoch...


Train:   0%|          | 0/123 [00:00<?, ?it/s]c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\torch\nn\functional.py:5504: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


[Train] Epoch complete. Avg Loss: 0.428579

[Val] Running validation...


c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")


[Val] Loss: 0.387663 | Pose: 0.026641 | Vel: 0.020643 | MPJPE: 3.2935 | VelErr: 1.4852 | GlossAcc: 0.9598 | LSMU: 0.8268 | 
Unique GT classes: 578
Unique Pred classes: 1
Total labels: 1000
[WARNING]: Model collapsing to very few classes!


KeyboardInterrupt: 

# 10 - testing

In [ ]:
import torch, json, os, numpy as np
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace

# ------------------
ckpt_path = "checkpoints_text2sign_3_3/best_text2sign.pth"
checkpoint = torch.load(ckpt_path, map_location="cpu")

cfg = checkpoint["cfg"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Run] device: {device}")

# -----------
tokenizer = HFTokenizer(texts=[], min_freq=1)

# restore vocab
tokenizer.itos = checkpoint["tokenizer_itos"]
tokenizer.stoi = {w: i for i, w in enumerate(tokenizer.itos)}

# rebuild tokenizer backend
tokenizer.tokenizer = Tokenizer(
    WordPiece(
        vocab=tokenizer.stoi,
        unk_token=tokenizer.unk_token
    )
)
tokenizer.tokenizer.pre_tokenizer = Whitespace()

# restore ids
tokenizer.pad_id = tokenizer.stoi[tokenizer.pad_token]
tokenizer.sos_id = tokenizer.stoi[tokenizer.sos_token]
tokenizer.eos_id = tokenizer.stoi[tokenizer.eos_token]
tokenizer.unk_id = tokenizer.stoi[tokenizer.unk_token]

print(f"[Tokenizer] vocab restored: {tokenizer.vocab_size}")

# Recover landmark dim

state_dict = checkpoint["model_state_dict"]
landmark_dim = state_dict["out_proj.weight"].shape[0]
print(f"[Landmark dim] {landmark_dim}")

# Build model

model = TextToSignModel(
    vocab_size=tokenizer.vocab_size,
    landmark_dim=landmark_dim,
    cfg=cfg
).to(device)

model.load_state_dict(state_dict)
model.eval()
print("==> Model loaded successfully")


# Inference

os.makedirs(cfg["save_dir"], exist_ok=True)

texts_to_generate = [
    "hello",
    "thank you",
    "how are you",
]

with torch.no_grad():
    for i, text in enumerate(texts_to_generate):
        preds = generate_from_text(
            model,
            tokenizer,
            text,
            cfg,
            device
        )  # (T, 225)

        # reshape back to (T, 75, 3) if needed
        preds = preds.reshape(cfg["max_landmark_len"], 75, 3)

        np.save(
            os.path.join(cfg["save_dir"], f"pred_{i}.npy"),
            preds.astype(np.float32)
        )

        np.savetxt(
            os.path.join(cfg["save_dir"], f"pred_{i}.csv"),
            preds.reshape(preds.shape[0], -1),
            delimiter=","
        )

        print(f"[OK] '{text}' → {preds.shape} saved")


[Run] device: cuda
[Tokenizer] vocab restored: 768
[Landmark dim] 225
==> Model loaded successfully
[OK] 'hello' → (20, 75, 3) saved
[OK] 'thank you' → (20, 75, 3) saved
[OK] 'how are you' → (20, 75, 3) saved


In [66]:
import numpy as np
import cv2
import mediapipe as mp
import os

INPUT_FILE = os.path.join(cfg['save_dir'], "pred_1.npy")  # (T,75,3)
OUTPUT_VIDEO = "ckp_3_2_pred_1_output.mp4"
FPS = 15
IMG_SIZE = 800

POSE = 33
HAND = 21

mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands

# POSE_CONN = list(mp_pose.POSE_CONNECTIONS)
# HAND_CONN = list(mp_hands.HAND_CONNECTIONS)

POSE_CONN = [
    (11,12),(11,13),(13,15),(12,14),(14,16),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(29,31),
    (24,26),(26,28),(28,30),(30,32)
]

HAND_CONN = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(17,18),(18,19),(19,20),
    (0,17)
]

def compute_global_transform(data, size=720, pad=80):
    """
    Compute ONE transform for the whole sequence
    so skeleton stays centered.
    """
    all_pts = data[:, :, :2].reshape(-1, 2)

    min_xy = all_pts.min(axis=0)
    max_xy = all_pts.max(axis=0)

    span = max_xy - min_xy
    span[span == 0] = 1e-6

    scale = (size - 2*pad) / max(span)

    # Centering offset
    center_after_scale = (min_xy + max_xy) / 2 * scale
    canvas_center = np.array([size/2, size/2])

    offset = canvas_center - center_after_scale

    return scale, offset



def transform_points(points, scale, offset):
    pts = points[:, :2] * scale + offset
    return pts.astype(int)

def draw(img, pts, connections, color):
    for i, j in connections:
        cv2.line(img, tuple(pts[i]), tuple(pts[j]), color, 2)

# --------

data = np.load(INPUT_FILE).astype(np.float32)
frames = data.shape[0]

scale, offset = compute_global_transform(data, size=IMG_SIZE)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    FPS,
    (IMG_SIZE, IMG_SIZE)
)

print("Rendering video...")

for t in range(frames):
    frame_img = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * 255

    lm = data[t]

    pts2d = transform_points(lm, scale, offset)

    pose = pts2d[:33]
    left = pts2d[33:54]
    right = pts2d[54:75]

    # Pose
    for a, b in POSE_CONN:
        cv2.line(frame_img, tuple(pose[a]), tuple(pose[b]), (0, 0, 255), 2)

    for p in pose:
        cv2.circle(frame_img, tuple(p), 3, (0, 0, 200), -1)

    # Hands
    for a, b in HAND_CONN:
        cv2.line(frame_img, tuple(left[a]), tuple(left[b]), (255, 0, 0), 2)
        cv2.line(frame_img, tuple(right[a]), tuple(right[b]), (0, 200, 0), 2)

    for p in left:
        cv2.circle(frame_img, tuple(p), 3, (200, 0, 0), -1)
    for p in right:
        cv2.circle(frame_img, tuple(p), 3, (0, 200, 0), -1)
    
    # draw(canvas, pose, POSE_CONNECTIONS, (0,255,0))
    # draw(canvas, left, HAND_CONNECTIONS, (255,0,0))
    # draw(canvas, right, HAND_CONNECTIONS, (0,0,255))

    writer.write(frame_img)

writer.release()
print(f"== Video saved to {OUTPUT_VIDEO}")


Rendering video...
== Video saved to ckp_3_2_pred_1_output.mp4
